# Explainable AI — Captum Deletion Challenge

**Your task:** choose and tune a Captum attribution method to minimise **deletion AUC**.
Explain the supplied, frozen PneumoniaMNIST classifier. Lower deletion scores rank first.
Use **32 practice images**, compare methods, then export maps for all **64 scoring images**.
Keep an experiment table and explain your final choice. Do not retrain the classifier.

Save a copy to Drive. A GPU is optional; this small model also works on CPU.
[Captum documentation](https://captum.ai/api/index.html).

Dataset attribution: [MedMNIST v2, Yang et al. (2023)](https://www.nature.com/articles/s41597-022-01721-8),
derived from [Kermany et al. (2018)](https://doi.org/10.1016/j.cell.2018.02.010).
PneumoniaMNIST is distributed under CC BY 4.0. Practice images use validation; scoring images use test.

In [ ]:
%pip install -q captum==0.9.0

In [ ]:
import hashlib
import importlib
from pathlib import Path
import random
import urllib.request

import numpy as np
import torch
from torch import nn
from captum.attr import IntegratedGradients, Saliency, InputXGradient, Occlusion, FeatureAblation, NoiseTunnel
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/examples/data"
SOURCE_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/backend/app/challenges/captum_deletion"
for name, url in [
    ("reference_model.npz", f"{DATA_URL}/captum_reference_model.npz"),
    ("explanation_data.npz", f"{DATA_URL}/captum_explanation_data.npz"),
    ("deletion_reference.py", f"{SOURCE_URL}/reference.py"),
]:
    urllib.request.urlretrieve(url, name)
MODEL_SHA256 = hashlib.sha256(Path("reference_model.npz").read_bytes()).hexdigest()
assert MODEL_SHA256 == "5dc78be0359c9cc341ed1554921a4c66d0b46934d20bd602a2f15e7b10e570e7", "Use the current notebook and model together."
with np.load("explanation_data.npz", allow_pickle=False) as data:
    ids, scoring_images, practice_images = data["ids"], data["images"], data["practice_images"]
assert scoring_images.shape == (64, 784) and practice_images.shape == (32, 784)
import deletion_reference
importlib.reload(deletion_reference)
print("Device:", device, "| Practice:", len(practice_images), "| Scoring:", len(scoring_images))

## Load the fixed model

The frozen classifier is a **784 → 64 → 32 → 2 ReLU MLP**, trained on the official training split.
The PyTorch model uses the same committed weights as the server's NumPy scorer.
Scale raw pixels by 255, then normalise using mean/std 0.5. Classes: **normal (0)** and **pneumonia (1)**.
Explain the probability of the original predicted class. Black pixels are **-1** in normalised space.

In [ ]:
class ReferenceClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(784, 64), nn.Linear(64, 32), nn.Linear(32, 2)])
        with np.load("reference_model.npz", allow_pickle=False) as weights, torch.no_grad():
            for i, layer in enumerate(self.layers):
                layer.weight.copy_(torch.from_numpy(weights[f"w{i}"].T.copy()))
                layer.bias.copy_(torch.from_numpy(weights[f"b{i}"].copy()))
        for parameter in self.parameters():
            parameter.requires_grad_(False)

    def forward(self, images):
        x = torch.relu(self.layers[0](images.flatten(1)))
        x = torch.relu(self.layers[1](x))
        return self.layers[2](x).softmax(1)

model = ReferenceClassifier().to(device).eval()

def tensor_images(raw):
    x = torch.tensor(raw, dtype=torch.float32, device=device).reshape(-1, 1, 28, 28)
    return (x / 255 - 0.5) / 0.5

with torch.no_grad():
    torch_p = model(tensor_images(scoring_images)).cpu().numpy()
np.testing.assert_allclose(torch_p, deletion_reference.predict(scoring_images), atol=1e-5, rtol=1e-4)
print("Fixed PyTorch model matches the server classifier.")

## Benchmark rules

Fix the target to each image's **original predicted class**. Rank pixels by attribution value,
largest first, then replace **28 pixels per step with black** (raw value 0).
Measure the same target probability at **29 points**, including both endpoints.
The score is trapezoidal area against the fraction deleted, averaged across images.
**Lower is better**; ties use row-major order. The score is in `[0, 1]`, and is not ROC AUC or
initial-confidence-normalised. Local scoring uses the server's reference implementation.
[RISE deletion metric](https://github.com/eclique/RISE/blob/master/evaluation.py).

## Your explanation settings — edit and experiment

Compare methods, then tune integration steps, occlusion windows/stride, feature groups and smoothing.
Try **signed**, **positive-only** and **absolute** attributions; these can produce different rankings.
Extend `explain_batch` with other Captum methods if you wish. Keep the model fixed and record settings honestly.

In [ ]:
METHODS_TO_COMPARE = ["saliency", "input_x_gradient", "integrated_gradients", "occlusion", "feature_ablation", "smoothgrad"]
IG_STEPS = 32
OCCLUSION_WINDOW = 5
OCCLUSION_STRIDE = 3
FEATURE_GROUP_SIZE = 4
NOISE_SAMPLES = 16
NOISE_STDEV = 0.10
MAP_MODE = "positive"  # try "signed" or "absolute"
EXPLANATION_BATCH_SIZE = 4


def explain_batch(inputs, targets, method):
    baseline = -1.0  # black raw pixels after normalisation
    if method == "integrated_gradients":
        attribution = IntegratedGradients(model).attribute(inputs, baselines=baseline,
            target=targets, n_steps=IG_STEPS, internal_batch_size=32)
    elif method == "saliency":
        attribution = Saliency(model).attribute(inputs, target=targets, abs=False)
    elif method == "input_x_gradient":
        attribution = InputXGradient(model).attribute(inputs, target=targets)
    elif method == "occlusion":
        attribution = Occlusion(model).attribute(inputs,
            sliding_window_shapes=(1, OCCLUSION_WINDOW, OCCLUSION_WINDOW),
            strides=(1, OCCLUSION_STRIDE, OCCLUSION_STRIDE),
            baselines=baseline, target=targets, perturbations_per_eval=16)
    elif method == "feature_ablation":
        rows, cols = torch.meshgrid(torch.arange(28, device=device), torch.arange(28, device=device), indexing="ij")
        groups = (28 + FEATURE_GROUP_SIZE - 1) // FEATURE_GROUP_SIZE
        mask = (rows // FEATURE_GROUP_SIZE) * groups + cols // FEATURE_GROUP_SIZE
        attribution = FeatureAblation(model).attribute(inputs,
            feature_mask=mask.reshape(1, 1, 28, 28), baselines=baseline,
            target=targets, perturbations_per_eval=16)
    elif method == "smoothgrad":
        attribution = NoiseTunnel(Saliency(model)).attribute(inputs, target=targets,
            nt_type="smoothgrad", nt_samples=NOISE_SAMPLES, nt_samples_batch_size=4,
            stdevs=NOISE_STDEV, abs=False)
    else:
        raise ValueError(f"Implement the Captum method: {method}")
    if MAP_MODE == "positive":
        attribution = attribution.clamp(min=0)
    elif MAP_MODE == "absolute":
        attribution = attribution.abs()
    elif MAP_MODE != "signed":
        raise ValueError("MAP_MODE must be positive, signed or absolute.")
    return attribution.detach().cpu().numpy().reshape(-1, 784)


def explain_images(raw, method):
    targets = deletion_reference.predict(raw).argmax(1)  # exactly the server's fixed targets
    maps = []
    for start in range(0, len(raw), EXPLANATION_BATCH_SIZE):
        inputs = tensor_images(raw[start:start + EXPLANATION_BATCH_SIZE])
        target = torch.tensor(targets[start:start + EXPLANATION_BATCH_SIZE], device=device)
        maps.append(explain_batch(inputs, target, method))
    return np.concatenate(maps)

In [ ]:
practice_results, practice_maps = [], {}
for method in METHODS_TO_COMPARE:
    maps = explain_images(practice_images, method)
    fractions, curves, areas, targets = deletion_reference.deletion_curves(practice_images, maps)
    practice_maps[method] = maps
    practice_results.append({"method": method, "deletion_auc": float(areas.mean())})
    print(f"{method:22s} practice deletion AUC: {areas.mean():.4f}")
practice_results.sort(key=lambda row: row["deletion_auc"])
best_method = practice_results[0]["method"]
print("Best practice method:", best_method, "| Map transform:", MAP_MODE)

## Record your experiments

| Captum method | Settings / map transform | Practice deletion AUC | What did you learn? |
|---|---|---:|---|
| Initial comparison | | | |
| Your next experiment | | | |
| Your final choice | | | |

Why did your chosen method perform best? What changes with absolute rather than positive maps?
A low score reflects this fixed black-pixel perturbation test; discuss what it does and does not show about explanation quality.

In [ ]:
METHOD = best_method  # choose after comparing methods and settings on practice images
chosen_maps = practice_maps[METHOD]
fractions, curves, areas, targets = deletion_reference.deletion_curves(practice_images, chosen_maps)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(practice_images[0].reshape(28, 28), cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Original: explains class {targets[0]}")
limit = max(float(np.abs(chosen_maps[0]).max()), 1e-15)
axes[1].imshow(chosen_maps[0].reshape(28, 28), cmap="coolwarm", vmin=-limit, vmax=limit)
axes[1].set_title(METHOD)
axes[2].plot(fractions, curves.mean(0))
axes[2].set(xlabel="Fraction deleted", ylabel="Original target probability", ylim=(0, 1),
            title=f"Practice deletion AUC: {areas.mean():.4f}")
for ax in axes[:2]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Generate scoring explanations and export

Export maps for every scoring image. The server computes deletion with its own fixed classifier;
no notebook-reported score, model weights or executable code is submitted. Keep the method label
under 128 characters and the full settings in your experiment table.

In [ ]:
scoring_maps = explain_images(scoring_images, METHOD)
assert scoring_maps.shape == (64, 784) and np.isfinite(scoring_maps).all()
fractions, scoring_curves, scoring_areas, targets = deletion_reference.deletion_curves(scoring_images, scoring_maps)
METHOD_DESCRIPTION = f"{METHOD}; {MAP_MODE}; IG={IG_STEPS}; window={OCCLUSION_WINDOW}; stride={OCCLUSION_STRIDE}; group={FEATURE_GROUP_SIZE}; noise={NOISE_SAMPLES}/{NOISE_STDEV}"
assert len(METHOD_DESCRIPTION) <= 128
np.savez_compressed("attributions.npz", ids=ids, attributions=scoring_maps,
                    model_sha256=np.array(MODEL_SHA256), method=np.array(METHOD_DESCRIPTION))
print(f"Local scoring deletion AUC: {scoring_areas.mean():.4f} (lower is better)")
print("Upload attributions.npz to your arena student page.")
try:
    from google.colab import files
    files.download("attributions.npz")
except ImportError:
    print("Outside Colab: use the notebook file browser.")

## Submit

Upload **attributions.npz**. The projector shows the leaderboard, mean deletion curve and attribution overlays.
Your best successful **lowest deletion score** determines your rank. Keep this `.ipynb` with your methods,
settings, results and explanations. Use Captum-generated maps; the class relies on honest method descriptions.